In [12]:
import numpy as np
import cv2
import pickle
import struct
import bz2
import os
import time
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from pydivsufsort import divsufsort

with open('phase3_results.pkl', 'rb') as f:
    p3 = pickle.load(f)

img = p3['img']
height, width = p3['height'], p3['width']
region_map = p3['region_map']
special_regions = p3['special_regions']
gradient_regions = p3['gradient_regions']
flat_regions = p3['flat_regions']

output_dir = "comic/phase4_output_4c"
os.makedirs(output_dir, exist_ok=True)
num_pixels = height * width

actual_regions = set(int(r) for r in np.unique(region_map[region_map >= 0]))
special_regions = set(int(r) for r in special_regions) & actual_regions
gradient_regions = {int(k): v for k, v in gradient_regions.items() if int(k) in actual_regions}
flat_set = set(int(r) for r in flat_regions) & actual_regions

categorized = special_regions | set(gradient_regions.keys()) | flat_set
uncategorized = actual_regions - categorized
flat_set |= uncategorized
flat_regions = list(flat_set)

assert (special_regions | set(gradient_regions.keys()) | flat_set) == actual_regions
print(f"Special: {len(special_regions)}  Gradient: {len(gradient_regions)}  Flat: {len(flat_regions)}")

Special: 11387  Gradient: 51  Flat: 2215


In [13]:
def build_palette(region_ids, region_map, img, ptol):
    """Build a color palette with tolerance-based merging."""
    region_avg_colors = {}
    for rid in region_ids:
        mask = (region_map == rid)
        if np.sum(mask) > 0:
            region_avg_colors[rid] = np.mean(img[mask], axis=0)

    palette = []
    region_to_palette = {}
    sorted_rids = sorted(region_ids, key=lambda r: np.sum(region_map == r), reverse=True)

    for rid in sorted_rids:
        if rid not in region_avg_colors:
            continue
        color = region_avg_colors[rid]
        best_idx = -1
        for idx, (pal_color, pal_count) in enumerate(palette):
            if np.sqrt(np.sum((pal_color - color) ** 2)) < ptol:
                best_idx = idx
                break
        if best_idx >= 0:
            pal_color, pal_count = palette[best_idx]
            new_color = (pal_color * pal_count + color) / (pal_count + 1)
            palette[best_idx] = (new_color, pal_count + 1)
            region_to_palette[rid] = best_idx
        else:
            palette.append((color, 1))
            region_to_palette[rid] = len(palette) - 1

    return palette, region_to_palette


main_region_ids = set(gradient_regions.keys()) | set(flat_regions)
palette, region_to_palette = build_palette(main_region_ids, region_map, img, ptol=3)
special_palette, special_region_to_palette = build_palette(special_regions, region_map, img, ptol=10)

print(f"Main palette: {len(palette)} colors")
print(f"Special palette: {len(special_palette)} colors")

Main palette: 783 colors
Special palette: 594 colors


In [14]:
def build_all_rle_runs(region_map, height, width):
    """
    Build a run-length list for every region in a single pass.
    Each run is (row, start_col, length): pixels
    region_map[row, start_col:start_col+length] all belong to that region.
    """
    runs_by_region = {}
    for y in range(height):
        row = region_map[y]
        x = 0
        while x < width:
            rid = int(row[x])
            if rid < 0:
                x += 1
                continue
            start = x
            while x < width and row[x] == rid:
                x += 1
            runs_by_region.setdefault(rid, []).append((y, start, x - start))
    return runs_by_region


all_regions = special_regions | set(gradient_regions.keys()) | set(flat_regions)
all_runs = build_all_rle_runs(region_map, height, width)

total_runs = sum(len(all_runs.get(rid, [])) for rid in all_regions)
print(f"RLE runs built for {len(all_regions)} regions")
print(f"Total runs: {total_runs}")

RLE runs built for 13653 regions
Total runs: 49468


In [15]:
def burrows_wheeler_transform_fast(data):
    """
    Sentinel-free Burrows-Wheeler Transform using the doubled-string trick:
    build a suffix array of (data+data) and keep only starting positions
    less than n, which represent the n cyclic rotations of data in
    correctly sorted order.
    """
    s = bytes(data)
    n = len(s)
    doubled = s + s

    sa_full = divsufsort(np.frombuffer(doubled, dtype=np.uint8).copy())
    sa_rotations = [i for i in sa_full if i < n]

    bwt_data = bytes([s[(i - 1) % n] for i in sa_rotations])
    bwt_index = sa_rotations.index(0)

    return bwt_data, bwt_index


def inverse_burrows_wheeler_transform(bwt_data, bwt_index):
    """
    Inverse Burrows-Wheeler Transform using the LF-mapping trick:
    sorting (char, original_index) pairs reproduces the correct
    next-pointer even with many repeated byte values.
    """
    L = bwt_data
    n = len(L)
    T = sorted(range(n), key=lambda i: (L[i], i))

    result = []
    idx = bwt_index
    for _ in range(n):
        idx = T[idx]
        result.append(L[idx])

    return bytes(result)

In [16]:
def move_to_front_transform(data):
    """Convert repeated symbols into runs of 0."""
    symbol_table = list(range(256))
    result = bytearray(len(data))

    for i, byte_val in enumerate(data):
        pos = symbol_table.index(byte_val)
        result[i] = pos
        symbol_table.pop(pos)
        symbol_table.insert(0, byte_val)

    return bytes(result)


def inverse_move_to_front_transform(data):
    """Inverse Move-To-Front Transform."""
    symbol_table = list(range(256))
    result = bytearray(len(data))

    for i, idx in enumerate(data):
        byte_val = symbol_table[idx]
        result[i] = byte_val
        symbol_table.pop(idx)
        symbol_table.insert(0, byte_val)

    return bytes(result)

In [17]:
def apply_rle_on_zeros(data):
    """Run-Length Encoding specialized for the zero runs produced by MTF."""
    result = bytearray()
    i = 0
    n = len(data)

    while i < n:
        if data[i] == 0:
            zero_count = 0
            while i < n and data[i] == 0 and zero_count < 254:
                zero_count += 1
                i += 1
            if zero_count == 1:
                result.append(0)
            else:
                result.append(255)
                result.append(zero_count)
        else:
            if data[i] == 255:
                result.append(255)
                result.append(255)
            else:
                result.append(data[i])
            i += 1

    return bytes(result)


def inverse_rle_on_zeros(data):
    """Inverse of apply_rle_on_zeros."""
    result = bytearray()
    i = 0
    n = len(data)

    while i < n:
        if data[i] == 255:
            if i + 1 < n:
                if data[i + 1] == 255:
                    result.append(255)
                    i += 2
                else:
                    zero_count = data[i + 1]
                    result.extend([0] * zero_count)
                    i += 2
            else:
                i += 1
        else:
            result.append(data[i])
            i += 1

    return bytes(result)

In [18]:
def apply_full_transformation(data):
    """Apply the full BWT -> MTF -> RLE pipeline."""
    bwt_data, bwt_index = burrows_wheeler_transform_fast(data)
    mtf_data = move_to_front_transform(bwt_data)
    rle_data = apply_rle_on_zeros(mtf_data)

    print(f"  Raw:        {len(data)} bytes")
    print(f"  After BWT:  {len(bwt_data)} bytes")
    print(f"  After MTF:  {len(mtf_data)} bytes")
    print(f"  After RLE:  {len(rle_data)} bytes")

    return rle_data, bwt_index


def inverse_full_transformation(data, bwt_index):
    """Inverse of apply_full_transformation: RLE -> MTF -> BWT."""
    rle_inv = inverse_rle_on_zeros(data)
    mtf_inv = inverse_move_to_front_transform(rle_inv)
    bwt_inv = inverse_burrows_wheeler_transform(mtf_inv, bwt_index)
    return bwt_inv

In [19]:
data = bytearray()

data.extend(struct.pack('>HH', height, width))

data.extend(struct.pack('>H', len(palette)))
for color, count in palette:
    r, g, b = np.clip(color, 0, 255).astype(np.uint8)
    data.extend(struct.pack('BBB', int(r), int(g), int(b)))

data.extend(struct.pack('>H', len(special_palette)))
for color, count in special_palette:
    r, g, b = np.clip(color, 0, 255).astype(np.uint8)
    data.extend(struct.pack('BBB', int(r), int(g), int(b)))

total_regions = len(special_regions) + len(gradient_regions) + len(flat_regions)
data.extend(struct.pack('>I', total_regions))

def write_runs_header(data, runs):
    data.extend(struct.pack('>I', len(runs)))
    for row, start_col, length in runs:
        data.extend(struct.pack('>HHH', row, start_col, length))

for rid in special_regions:
    runs = all_runs.get(rid, [])
    pal_idx = special_region_to_palette.get(rid, 0)
    data.extend(struct.pack('B', 2))
    data.extend(struct.pack('>H', pal_idx))
    write_runs_header(data, runs)

for rid, grad_info in gradient_regions.items():
    runs = all_runs.get(rid, [])
    direction = grad_info['direction']
    start_color = np.clip(grad_info['start_color'], 0, 255).astype(np.uint8)
    end_color = np.clip(grad_info['end_color'], 0, 255).astype(np.uint8)
    data.extend(struct.pack('B', 1))
    data.extend(struct.pack('B', int(direction)))
    data.extend(struct.pack('BBB', *start_color.tolist()))
    data.extend(struct.pack('BBB', *end_color.tolist()))
    write_runs_header(data, runs)

for rid in flat_regions:
    runs = all_runs.get(rid, [])
    pal_idx = region_to_palette.get(rid, 0)
    data.extend(struct.pack('B', 0))
    data.extend(struct.pack('>H', pal_idx))
    write_runs_header(data, runs)

raw_data = bytes(data)
print(f"Raw encoded data: {len(raw_data)} bytes")

Raw encoded data: 396777 bytes


In [20]:
compressed = bz2.compress(raw_data, 9)
size = len(compressed)
bpp = size * 8 / num_pixels
print(f"Compressed size: {size} bytes | BPP: {bpp:.4f}")

Compressed size: 178850 bytes | BPP: 4.5509


In [21]:
phase4_results = {
    'img': img,  # only for comparison in Phase 5, not used by the decoder
    'height': height,
    'width': width,
    'compressed_data': compressed,
    'size': size,
    'bpp': bpp
}
with open('phase4_results.pkl', 'wb') as f:
    pickle.dump(phase4_results, f)
print("Saved phase4_results.pkl")

Saved phase4_results.pkl


In [22]:
# Per-type byte breakdown of the raw stream (read-only diagnostic, changes nothing)
def runs_bytes(runs):
    return 4 + 6 * len(runs)          # run count (4 B) + 6 B per run

def runs_pixels(runs):
    return sum(r[2] for r in runs)

stats = {}

# header byte sizes per region type (same as the encoder above)
HEADER = {'special': 1 + 2, 'gradient': 1 + 1 + 3 + 3, 'flat': 1 + 2}

groups = {
    'special': list(special_regions),
    'gradient': list(gradient_regions.keys()),
    'flat': list(flat_regions),
}

print(f"{'type':9s} {'regions':>8s} {'runs':>8s} {'raw bytes':>10s} {'share':>7s} {'pixels':>8s} {'px share':>9s}")
total_raw = len(raw_data)
for name, ids in groups.items():
    n_runs = sum(len(all_runs.get(r, [])) for r in ids)
    raw_b = sum(HEADER[name] + runs_bytes(all_runs.get(r, [])) for r in ids)
    px = sum(runs_pixels(all_runs.get(r, [])) for r in ids)
    stats[name] = (len(ids), n_runs, raw_b, px)
    print(f"{name:9s} {len(ids):8d} {n_runs:8d} {raw_b:10d} {raw_b/total_raw:7.1%} {px:8d} {px/num_pixels:9.1%}")

pal_bytes = 2 + 3 * len(palette) + 2 + 3 * len(special_palette)
print(f"\npalettes + file header: {pal_bytes + 4 + 4} bytes ({(pal_bytes + 8)/total_raw:.1%})")

# Special regions by size: where do the bytes go?
print("\nSpecial regions by pixel count:")
bins = [(1, 2), (3, 9), (10, 49), (50, 10**9)]
for lo, hi in bins:
    ids = [r for r in groups['special']
           if lo <= runs_pixels(all_runs.get(r, [])) <= hi]
    raw_b = sum(HEADER['special'] + runs_bytes(all_runs.get(r, [])) for r in ids)
    px = sum(runs_pixels(all_runs.get(r, [])) for r in ids)
    label = f"{lo}-{hi}" if hi < 10**9 else f"{lo}+"
    print(f"  {label:>6s} px: {len(ids):6d} regions, {raw_b:8d} B ({raw_b/total_raw:5.1%}), {px/num_pixels:5.1%} of image")

# bz2 size of each type compressed on its own (approximation of its real cost)
print("\nbz2 size when each type is compressed separately:")
for name, ids in groups.items():
    buf = bytearray()
    for r in ids:
        runs = all_runs.get(r, [])
        buf.extend(b'\x00' * HEADER[name])
        write_runs_header(buf, runs)
    print(f"  {name:9s} {len(bz2.compress(bytes(buf), 9)):8d} B")

type       regions     runs  raw bytes   share   pixels  px share
special      11387    30127     260471   65.6%    54942     17.5%
gradient        51     3507      21654    5.5%    77109     24.5%
flat          2215    15834     110509   27.9%   182349     58.0%

palettes + file header: 4143 bytes (1.0%)

Special regions by pixel count:
     1-2 px:   4612 regions,    74002 B (18.7%),  2.9% of image
     3-9 px:   5854 regions,   132814 B (33.5%),  8.1% of image
   10-49 px:    853 regions,    44923 B (11.3%),  4.8% of image
     50+ px:     68 regions,     8732 B ( 2.2%),  1.6% of image

bz2 size when each type is compressed separately:
  special      89163 B
  gradient     10443 B
  flat         50714 B
